# DISCRETIO — Object Replacement & Counterfactual Pipeline on Kaggle

This notebook tests and compares the two object replacement methods for the DISCRETIO benchmark on 10 IDD-PeD child pedestrian clips:
- **Method 1**: Two-stage Modular (SAM 2 Mask $	o$ ProPainter Background Plate $	o$ DWPose Retarget $	o$ Pose-guided insertion)
- **Method 2**: One-stage Unified (SAM 2 Mask $	o$ Expanded Bounding Box $	o$ Inpainting diffusion)

**Settings Required:**
- Accelerator: **GPU T4 x2** or **P100**
- Internet: **ON**


In [ ]:
# 1. Hardware & GPU Check
!nvidia-smi
!df -h /kaggle/working



In [ ]:
# 2. Locate Dataset (No unzipping needed — Kaggle datasets are pre-extracted)
import os, json
from pathlib import Path
import cv2
import pandas as pd

WORKDIR = Path("/kaggle/working/discretio")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# Locate manifest in /kaggle/input
manifests = list(Path("/kaggle/input").rglob("selected_clips.json"))
assert len(manifests) >= 1, "selected_clips.json not found in /kaggle/input! Make sure child_clips is attached."

CLIPS_JSON = manifests[0]
VIDEO_DIR = manifests[0].parent

print("✓ CLIPS_JSON =", CLIPS_JSON)
print("✓ VIDEO_DIR  =", VIDEO_DIR)

# Validate 10 clips
clips = json.loads(CLIPS_JSON.read_text())
assert len(clips) == 10, f"Expected 10 clips, found {len(clips)}?"
assert all(c["age_bracket"] == "child" for c in clips), "Manifest includes non-child clips."

rows = []
for c in clips:
    video = next(VIDEO_DIR.rglob(c["local_video"]), None)
    assert video is not None, f"Missing video: {c['local_video']}"
    cap = cv2.VideoCapture(str(video))
    assert cap.isOpened(), f"Cannot open: {video}"
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    assert frame_count == 151, f"{video.name}: expected 151 frames, got {frame_count}"
    rows.append({
        "clip_id": c["clip_id"],
        "video": video.name,
        "age": c["age_bracket"],
        "frames": frame_count,
        "size_MB": round(video.stat().st_size / 1e6, 1),
        "resolution": f"{width}x{height}",
        "outcome": c["outcome"],
    })

display(pd.DataFrame(rows))
print("✓ Distinct source videos:", len({c["video_id"] for c in clips}))


In [ ]:
# 3. Clone ProPainter & Download Model Checkpoints
!git clone https://github.com/sczhou/ProPainter.git /kaggle/working/ProPainter
%cd /kaggle/working/ProPainter
!mkdir -p weights
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/ProPainter.pth -P weights/
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/recurrent_flow_completion.pth -P weights/
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/raft-things.pth -P weights/

# Install ProPainter dependencies
!pip install -q -r requirements.txt
%cd /kaggle/working/discretio



In [ ]:
# 4. Install SAM 2 and Core Evaluation Metrics
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q lpips opencv-python Pillow pandas scipy

# Download SAM 2 Large checkpoint
!mkdir -p /kaggle/working/checkpoints
!wget -nc --show-progress https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt -P /kaggle/working/checkpoints/



In [ ]:
# 5. Extract Code Pipeline if uploaded as zip, or verify files
code_zips = glob.glob('/kaggle/input/**/counterfactual_pipeline.zip', recursive=True)
if code_zips:
    !unzip -q -o {code_zips[0]} -d /kaggle/working/discretio/
    print('✓ Unzipped counterfactual_pipeline.zip')
else:
    print('Assuming counterfactual_pipeline files are present in working directory.')



In [ ]:
# 6. PILOT TEST: Run Step 1 (SAM 2) & Step 2 (ProPainter) on a SINGLE Clip
PILOT_CLIP = 'IDD-PeD_gp_set_0006_vid_0003__gp_6959__t02510'

%cd /kaggle/working/discretio/counterfactual_pipeline
!python 01_segment_pedestrian.py     --clips_json {CLIPS_JSON}     --video_dir {VIDEO_DIR}     --out_dir /kaggle/working/pipeline_output/01_masks     --sam2_checkpoint /kaggle/working/checkpoints/sam2_hiera_large.pt     --sam2_config sam2_hiera_l.yaml     --clip_ids {PILOT_CLIP}

!python 02_erase_pedestrian.py     --clips_json {CLIPS_JSON}     --masks_dir /kaggle/working/pipeline_output/01_masks     --video_dir {VIDEO_DIR}     --out_dir /kaggle/working/pipeline_output/02_erased     --propainter_dir /kaggle/working/ProPainter     --clip_ids {PILOT_CLIP}



In [ ]:
# 7. VISUAL INSPECTION: View Intermediate Results (Mask + Erased Plate)
import sys
sys.path.append('/kaggle/working/discretio/counterfactual_pipeline')
from visualize_comparison import export_keyframe_grid, export_clip_comparison
from IPython.display import Image, display

grid_path = export_keyframe_grid(
    PILOT_CLIP,
    pipeline_output_dir='/kaggle/working/pipeline_output',
    out_dir='/kaggle/working/pipeline_output/visualizations',
    n_frames=3,
    zoom_crop=True
)

print('Keyframe Progression Grid (Original vs SAM 2 Mask vs ProPainter Erased Plate):')
display(Image(filename=grid_path))



In [ ]:
# 8. Run Object Replacement: Method 1 (Pose Retarget) & Method 2 (Unified Inpaint)
# For testing the object replacement interventions:
!python 03a_pose_retarget.py     --clips_json {CLIPS_JSON}     --masks_dir /kaggle/working/pipeline_output/01_masks     --erased_dir /kaggle/working/pipeline_output/02_erased     --video_dir {VIDEO_DIR}     --out_dir /kaggle/working/pipeline_output/03a_method1     --target_age adult     --clip_ids {PILOT_CLIP}

!python 03b_unified_inpaint.py     --clips_json {CLIPS_JSON}     --masks_dir /kaggle/working/pipeline_output/01_masks     --video_dir {VIDEO_DIR}     --out_dir /kaggle/working/pipeline_output/03b_method2     --target_age adult     --clip_ids {PILOT_CLIP}



In [ ]:
# 9. Automated Metrics & Comparison Table
!python 05_metrics_auto.py     --clips_json {CLIPS_JSON}     --method1_dir /kaggle/working/pipeline_output/03a_method1     --method2_dir /kaggle/working/pipeline_output/03b_method2     --masks_dir /kaggle/working/pipeline_output/01_masks     --video_dir {VIDEO_DIR}     --out_dir /kaggle/working/pipeline_output/05_metrics     --target_age adult     --clip_ids {PILOT_CLIP}

# Display report
with open('/kaggle/working/pipeline_output/05_metrics/method_comparison.txt') as f:
    print(f.read())



In [ ]:
# 10. Side-by-Side 5-Panel Visual Comparison Video / GIF
_, gif_path = export_clip_comparison(
    PILOT_CLIP,
    pipeline_output_dir='/kaggle/working/pipeline_output',
    out_dir='/kaggle/working/pipeline_output/visualizations',
    fps=15,
    zoom_crop=True,
    max_frames=45
)
display(Image(filename=gif_path))

